# Retirement Tax Optimizer — Spouse A & Spouse B (package edition)

**Goal:** maximize after-tax terminal net worth at the planning horizon, subject to never running out of money, by jointly choosing:

1. **Pre-retirement** allocation between Traditional 401(k) and Roth 401(k) for each spouse.
2. **In-retirement** withdrawal sequencing across taxable / pretax / Roth buckets.
3. **Roth-conversion** size during the retirement-to-RMD gap years.

All simulation logic lives in the `tax_optimizer/` Python package. This notebook is a thin orchestration layer that:

- Imports the engine and configures a scenario.
- Runs the four-strategy comparison + tornado sensitivity (deterministic).
- Demonstrates the v2 capabilities:
  - **Monte Carlo** sequence-of-returns risk (`§8`).
  - **Widow's-penalty / single-filer mortality** stress test (`§9`).
  - **Tax-regime change** (TCJA sunset) stress test (`§10`).
  - **Smile-shaped retirement spending** + lump events (`§11`).
  - **Asset location** (per-account equity / bond mix) (`§12`).

To use your own scenario, edit cell `§1` and re-run.


## §1. Scenario inputs

In [ ]:
from __future__ import annotations
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tax_optimizer import (
    Config,
    Inputs,
    StartingBalances,
    CurrentIncome,
    CurrentContrib,
    PensionInputs,
    SocialSecurity,
    SpendingProfile,
    SpendingPhase,
    LumpEvent,
    LongTermCareShock,
    AssetLocation,
    AssetMix,
    DeterministicModel,
    LognormalModel,
    BootstrapModel,
    Mortality,
    TCJA_EXTENDED,
    PRE_TCJA_2017,
    SUNSET_2026,
    simulate,
    simulate_paths,
    summarize,
    optimize_s3,
    BRACKET_CHOICES,
    tornado_sensitivity,
    render_actions,
    render_takeaways,
)
from tax_optimizer.tax import federal_tax, irmaa_annual_surcharge

# -- Scenario: realistic dual-income MFJ couple turning 50 in 2026 -----------
inputs = Inputs(
    starting=StartingBalances(
        spouse_a_pretax_401k=225_000.0,
        spouse_b_pretax_401k=150_000.0,
        spouse_a_roth_ira=40_000.0,
        spouse_b_pretax_ira=35_000.0,
        pension_balance=0.0,
        hsa=18_000.0,
        taxable_brokerage=80_000.0,
    ),
    income=CurrentIncome(
        spouse_a_gross=95_000.0,
        spouse_b_gross=70_000.0,
        spouse_a_bonus=5_000.0,
        interest=500.0,
        capital_gains=1_000.0,
        dividends=2_000.0,
    ),
    contrib=CurrentContrib(
        spouse_a_pct=0.08,
        spouse_b_pct=0.06,
        spouse_a_roth_pct=0.0,
        spouse_b_roth_pct=0.0,
        hsa_family=8_550.0,
        std_deduction=32_200.0,
        baseline_tax=0.0,
    ),
    pension=PensionInputs(balance_today=0.0, monthly_at_nrd=0.0),
    ss=SocialSecurity(monthly_spouse_a=2_700.0, monthly_spouse_b=2_200.0),
    annual_expenses=85_000.0,
)

# -- Default Config: deterministic 6% growth, MFJ, 50 -> 90, retire at 65 ----
cfg = Config(
    spouse_a_total_contrib_pct=inputs.contrib.spouse_a_pct,
    spouse_b_total_contrib_pct=inputs.contrib.spouse_b_pct,
    spouse_a_roth_401k_pct=inputs.contrib.spouse_a_roth_pct,
    spouse_b_roth_401k_pct=inputs.contrib.spouse_b_roth_pct,
    annual_expenses_today=inputs.annual_expenses,
    nominal_growth_rate=0.06,
    inflation=0.025,
    wage_growth=0.030,
)

print('--- Starting balances ---')
for k, v in inputs.starting.__dict__.items():
    print(f'  {k:30s} ${v:>14,.0f}')
print(f'\nAnnual expenses:    ${inputs.annual_expenses:,.0f}')
print(f'Horizon:           {cfg.spouse_a_age_start} -> {cfg.horizon_age}')
print(f'Tax regime:        {cfg.tax_regime.name}')


## §2. Sanity check — first-year federal tax

Verify the package's federal-tax engine produces sensible numbers
against a hand calculation for the scenario's first working year.


In [ ]:
# First-year wages, pre-tax 401(k) deferrals reduce wages_box1.
a_pretax = inputs.income.spouse_a_gross * inputs.contrib.spouse_a_pct
b_pretax = inputs.income.spouse_b_gross * inputs.contrib.spouse_b_pct
wages_box1 = (
    inputs.income.spouse_a_gross + inputs.income.spouse_a_bonus
    + inputs.income.spouse_b_gross
    - a_pretax - b_pretax
)

ftax = federal_tax(
    regime=cfg.tax_regime,
    filing_status='mfj',
    wages=wages_box1,
    interest=inputs.income.interest,
    qualified_div=inputs.income.dividends,
    ltcg=inputs.income.capital_gains,
)
print(f'Year 1 AGI:           ${ftax["agi"]:,.0f}')
print(f'Year 1 taxable inc:   ${ftax["taxable_income"]:,.0f}')
print(f'Year 1 federal tax:   ${ftax["tax"]:,.0f}')
print(f'Year 1 marginal rate: {ftax["marginal"]:.0%}')


## §3. Strategy comparison: S0 / S1 / S2 / S3

Four candidate strategies, all evaluated deterministically (single
6%-growth path). The optimizer (`S3`) uses `differential_evolution`
because the IRMAA cliffs make the objective non-smooth.


In [ ]:
s0_cfg = cfg
s1_cfg = replace(cfg, spouse_a_roth_401k_pct=1.0, spouse_b_roth_401k_pct=1.0)
s2_cfg = replace(cfg, roth_conversion_target_bracket=0.22)
s3_cfg, x_opt = optimize_s3(cfg, inputs, objective='terminal')
print(f'S3 optimizer chose:  Spouse A Roth %={s3_cfg.spouse_a_roth_401k_pct:.2f}  '
      f'Spouse B Roth %={s3_cfg.spouse_b_roth_401k_pct:.2f}  '
      f'Conv bracket={s3_cfg.roth_conversion_target_bracket:.0%}')

results = {}
for name, c in [
    ('S0_baseline', s0_cfg),
    ('S1_all_roth_401k', s1_cfg),
    ('S2_bracket_fill_22', s2_cfg),
    ('S3_optimized', s3_cfg),
]:
    df = simulate(c, inputs)
    results[name] = (c, df, summarize(df))

summary_df = pd.DataFrame({n: s for n, (_c, _d, s) in results.items()}).T
summary_df[['lifetime_tax_npv', 'lifetime_irmaa_npv', 'terminal_after_tax', 'peak_marginal', 'years_irmaa']]


## §4. Visualizations: balances, taxes, contributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for name, (_c, df, _s) in results.items():
    liquid = df['pretax_balance'] + df['roth_balance'] + df['taxable_balance']
    axes[0,0].plot(df['spouse_a_age'], liquid, label=name)
    axes[0,1].plot(df['spouse_a_age'], df['federal_tax'], label=name)
    axes[1,0].plot(df['spouse_a_age'], df['pretax_balance'], label=name, linestyle='-')
    axes[1,0].plot(df['spouse_a_age'], df['roth_balance'], label=f'{name} (Roth)', linestyle='--')
    axes[1,1].plot(df['spouse_a_age'], df['agi'], label=name)
axes[0,0].set_title('Liquid balance (pretax + Roth + taxable)')
axes[0,1].set_title('Federal tax / year')
axes[1,0].set_title('Pretax (solid) vs Roth (dashed)')
axes[1,1].set_title('AGI / year')
for ax in axes.flat:
    ax.legend(fontsize=7)
    ax.set_xlabel('Spouse A age')
plt.tight_layout()
plt.show()


## §5. Year-by-year detail of the winning strategy

Each row is one simulation year. Key columns:
- `wages` / `pension` / `ssn` — gross income items.
- `rmd_a` / `rmd_b` — required minimum distributions (per IRS Uniform Lifetime).
- `roth_conversion` — pretax → Roth conversion in gap years.
- `pretax_withdrawal` / `taxable_withdrawal` — strategy-driven retirement draws.
- `agi` / `federal_tax` — IRC §1 computation for the active filing status.
- `irmaa` / `irmaa_tier` — Medicare premium surcharge (Tier 0 = no surcharge).
- `marginal` — top federal ordinary bracket reached.
- `*_balance` columns — end-of-year liquid balances per bucket.
- `equity_return` / `bond_return` — that year's draws from the active `MarketModel`
  (constant for `DeterministicModel`, stochastic otherwise).


In [ ]:
winner_name = max(results, key=lambda n: results[n][2]['terminal_after_tax'])
winner_df = results[winner_name][1]
print(f'Winning strategy: {winner_name}')

show_cols = [
    'year','spouse_a_age','spouse_b_age','filing_status','wages','pension','ssn',
    'rmd_a','rmd_b','roth_conversion','pretax_withdrawal','taxable_withdrawal',
    'agi','federal_tax','irmaa','irmaa_tier','marginal',
    'pretax_a_balance','pretax_b_balance','roth_balance','taxable_balance',
]
fmt = {col: '${:>12,.0f}' for col in show_cols if col not in
       ('year','spouse_a_age','spouse_b_age','filing_status','irmaa_tier')}
fmt['marginal'] = '{:.0%}'
fmt['irmaa_tier'] = '{:>2.0f}'
winner_df[show_cols].iloc[15:].style.format(fmt)


## §6. Tornado sensitivity

In [ ]:
sens_df, base_terminal = tornado_sensitivity(cfg, inputs)
print(f'Base terminal NW: ${base_terminal:,.0f}\n')

fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(sens_df))
ax.barh(y, sens_df['delta_high'].values / 1e3, alpha=0.7, label='high - base')
ax.barh(y, sens_df['delta_low'].values / 1e3, alpha=0.7, label='low - base')
ax.set_yticks(y); ax.set_yticklabels(sens_df['param'].str.replace('_', ' '))
ax.invert_yaxis(); ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Change in terminal NW ($K)')
ax.set_title('Tornado: parameter sensitivity'); ax.legend()
plt.tight_layout(); plt.show()
sens_df.head(8)


## §7. Recommended actions + takeaways

In [ ]:
print(render_takeaways(results, cfg))
print()
print(render_actions(results, sens_df, cfg, base_terminal))


## §8. Monte Carlo — sequence-of-returns risk

The deterministic simulator above assumes a flat 6%/yr return every year.
That hides "sequence-of-returns risk": a market crash early in retirement
hurts a withdrawing portfolio far more than the same crash late in life.

Below we re-run the same `cfg` with a `LognormalModel` (independent yearly
draws, μ=7%, σ=18% for equities) and a `BootstrapModel` (block-bootstrap
from 1928–2023 historical S&P + 10y-Treasury data). The summary reports:

- `prob_success` — fraction of paths whose liquid balance never falls
  below that year's spending need.
- `terminal_p5/p50/p95` — 5th / median / 95th percentile of terminal
  after-tax NW.
- `cvar_terminal_p10` — average terminal NW across the worst-10% paths
  (the metric a risk-averse retiree should actually optimize on).

Use `simulate_paths(..., n_paths=2000)` for tighter tails; the 200-path
demo below runs in a few seconds.


In [ ]:
cfg_lognormal = replace(cfg, market=LognormalModel(equity_mu=0.07, equity_sigma=0.18,
                                                  bond_mu=0.04, bond_sigma=0.06))
cfg_bootstrap = replace(cfg, market=BootstrapModel(block_size=5))

mc_lognormal = simulate_paths(cfg_lognormal, inputs, n_paths=200, seed=42)
mc_bootstrap = simulate_paths(cfg_bootstrap, inputs, n_paths=200, seed=42)

mc_summary = pd.DataFrame({
    'lognormal':  mc_lognormal.summary(),
    'bootstrap':  mc_bootstrap.summary(),
}).T
mc_summary


In [ ]:
# Fan chart of liquid balance across all 200 paths.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mc, title in [(axes[0], mc_lognormal, 'Lognormal'),
                      (axes[1], mc_bootstrap, 'Bootstrap (1928-2023)')]:
    ages = mc.paths[0]['spouse_a_age'].to_numpy()
    arr = np.stack([
        (df['pretax_balance'] + df['roth_balance'] + df['taxable_balance']).to_numpy()
        for df in mc.paths
    ]) / 1e6
    p5, p25, p50, p75, p95 = [np.percentile(arr, p, axis=0) for p in (5,25,50,75,95)]
    ax.fill_between(ages, p5, p95, alpha=0.15, label='5-95%')
    ax.fill_between(ages, p25, p75, alpha=0.30, label='25-75%')
    ax.plot(ages, p50, lw=2, label='median')
    ax.set_title(f'{title}: liquid balance fan ({mc.n_paths} paths)')
    ax.set_xlabel('Spouse A age'); ax.set_ylabel('Liquid ($M)'); ax.legend()
plt.tight_layout(); plt.show()


## §9. Widow's-penalty stress test

Realistic plans must stress the case where one spouse dies and the
survivor switches from MFJ to single-filer status. The single-filer
penalty:

- Brackets compress (the 22% bracket caps at ~$103k for single vs $207k MFJ).
- Standard deduction halves.
- IRMAA thresholds for Medicare premiums roughly halve.
- Survivor receives only the *larger* of the two SS checks, not both.
- Pension annuity scales by the joint-and-survivor election (default 50%).

Below we model Spouse A dying at year 25 (age 75) and re-run S3.


In [ ]:
cfg_widow = replace(s3_cfg, mortality=Mortality(year_of_death_a=25,
                                               pension_survivor_pct=0.5))

df_baseline = results['S3_optimized'][1]
df_widow = simulate(cfg_widow, inputs)

cmp = pd.DataFrame({
    'baseline (both alive)': summarize(df_baseline),
    'widow (A dies year 25)': summarize(df_widow),
}).T
cmp


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
liq_b = df_baseline['pretax_balance'] + df_baseline['roth_balance'] + df_baseline['taxable_balance']
liq_w = df_widow['pretax_balance'] + df_widow['roth_balance'] + df_widow['taxable_balance']
axes[0].plot(df_baseline['spouse_a_age'], liq_b/1e6, label='baseline')
axes[0].plot(df_widow['spouse_a_age'], liq_w/1e6, label='widow @y25')
axes[0].axvline(75, color='red', ls=':', alpha=0.6, label='death year')
axes[0].set_title('Liquid balance ($M)'); axes[0].set_xlabel('Spouse A age'); axes[0].legend()
axes[1].plot(df_baseline['spouse_a_age'], df_baseline['marginal']*100, label='baseline')
axes[1].plot(df_widow['spouse_a_age'], df_widow['marginal']*100, label='widow @y25')
axes[1].axvline(75, color='red', ls=':', alpha=0.6)
axes[1].set_title('Federal marginal bracket (%)'); axes[1].set_xlabel('Spouse A age'); axes[1].legend()
plt.tight_layout(); plt.show()


## §10. Tax-regime change — TCJA sunset stress test

The current `TCJA_EXTENDED` regime assumes Congress extends TCJA past
its scheduled 2025 expiration. If TCJA actually expires, brackets revert
to roughly the pre-TCJA structure with widths inflation-adjusted forward
(~1.30× the 2017 nominals; standard deduction roughly halves).

We model this with `regime_change_year_offset` + `regime_change_target`,
swapping to `SUNSET_2026` at year 5 (i.e. starting in 2031).


In [ ]:
cfg_sunset = replace(s3_cfg, regime_change_year_offset=5,
                                    regime_change_target=SUNSET_2026)
df_sunset = simulate(cfg_sunset, inputs)

# Re-optimize S3 *under* the sunset assumption to see how the optimizer
# responds — typically conversions become more attractive in years before
# the regime change.
cfg_sunset_base = replace(cfg, regime_change_year_offset=5,
                                regime_change_target=SUNSET_2026)
s3_sunset_cfg, _ = optimize_s3(cfg_sunset_base, inputs, objective='terminal')
df_sunset_opt = simulate(s3_sunset_cfg, inputs)

cmp = pd.DataFrame({
    'baseline (TCJA forever)':       summarize(df_baseline),
    'sunset @y5 (S3 unchanged)':     summarize(df_sunset),
    'sunset @y5 (re-optimized S3)':  summarize(df_sunset_opt),
}).T
cmp


In [ ]:
print(f'Original S3 conv target:        {s3_cfg.roth_conversion_target_bracket:.0%}')
print(f'Re-optimized under sunset:      {s3_sunset_cfg.roth_conversion_target_bracket:.0%}')
print(f'Original S3 Roth-401(k) (A/B):  {s3_cfg.spouse_a_roth_401k_pct:.0%} / {s3_cfg.spouse_b_roth_401k_pct:.0%}')
print(f'Re-optimized Roth-401(k) (A/B): {s3_sunset_cfg.spouse_a_roth_401k_pct:.0%} / {s3_sunset_cfg.spouse_b_roth_401k_pct:.0%}')


## §11. Smile-shaped retirement spending + lump events

Real spending follows the Blanchett "retirement smile":

- 100% of base in working years
- 115% of base in the "go-go" years (65–74; travel, hobbies)
- 95% of base in the "slow-go" years (75–84)
- 100% in the "no-go" years (85+) plus an LTC shock in the last 3
  years ($80k/yr today's dollars).

Plus we'll model a $50k wedding gift in year 15 and a $60k new car in year 22.


In [ ]:
smile = SpendingProfile.retirement_smile(
    base_spending=inputs.annual_expenses, inflation=0.025,
    ltc_years=3, ltc_annual_today=80_000.0,
)
smile.lump_events = [
    LumpEvent(year_offset=15, amount_today=50_000.0, label='kid wedding'),
    LumpEvent(year_offset=22, amount_today=60_000.0, label='replace car'),
]

cfg_smile = replace(s3_cfg, spending=smile)
df_smile = simulate(cfg_smile, inputs)

cmp = pd.DataFrame({
    'flat $85k (baseline S3)':            summarize(df_baseline),
    'smile + LTC + 2 lump events':        summarize(df_smile),
}).T
cmp


In [ ]:
# Show the actual spending need year-by-year so the smile shape is visible.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_baseline['spouse_a_age'], df_baseline['spending_need']/1e3, label='flat (baseline)')
ax.plot(df_smile['spouse_a_age'],    df_smile['spending_need']/1e3,    label='smile + events')
ax.set_title('Annual spending need ($K, nominal)')
ax.set_xlabel('Spouse A age'); ax.set_ylabel('$K'); ax.legend()
plt.tight_layout(); plt.show()


## §12. Asset location — bonds in pretax, equities in Roth

By default the package's `AssetLocation()` puts:

- 40% equity / 60% bond in the **pretax** sleeve  — bonds get sheltered
  from the ordinary-income tax line they would otherwise feed.
- 100% equity in the **Roth** sleeve              — maximize tax-free compounding.
- 80% equity / 20% bond in the **taxable** sleeve.
- 80% equity / 20% bond in the **HSA** sleeve.

Pair that with a `LognormalModel` so equity and bond returns differ
year-to-year, and the asset-location effect becomes visible.


In [ ]:
cfg_asset_uniform = replace(s3_cfg, market=LognormalModel(),
                                  asset_location=AssetLocation.uniform(equity_pct=0.6))
cfg_asset_located = replace(s3_cfg, market=LognormalModel(),
                                  asset_location=AssetLocation())   # bonds-in-pretax default

mc_uniform = simulate_paths(cfg_asset_uniform, inputs, n_paths=200, seed=7)
mc_located = simulate_paths(cfg_asset_located, inputs, n_paths=200, seed=7)

cmp = pd.DataFrame({
    'uniform 60/40 everywhere':           mc_uniform.summary(),
    'asset-located (bonds in pretax)':    mc_located.summary(),
}).T
cmp


## Recap — what changed in v2

| Capability | Where to set it | Effect on output |
|---|---|---|
| Stochastic returns | `cfg.market = LognormalModel(...)` | adds `prob_success`, percentile / CVaR metrics |
| Mortality / widow's penalty | `cfg.mortality = Mortality(year_of_death_a=...)` | flips filing status to single after death; SS / pension scale per election |
| Asset location | `cfg.asset_location = AssetLocation(...)` | per-account equity/bond split — bonds in pretax, equities in Roth |
| Smile spending + lumps | `cfg.spending = SpendingProfile.retirement_smile(...)` | per-age multipliers + lump events + LTC shock |
| Tax regime swap | `cfg.tax_regime`, `cfg.regime_change_year_offset` + `regime_change_target` | per-year regime selection |
| Optimizer objective | `optimize_s3(cfg, inputs, objective='cvar')` | CVaR or `'p_success'` instead of point-estimate terminal NW |

The `tax_optimizer/` package is the single source of truth — every cell
in this notebook imports from it. Run `python -m tax_optimizer --help`
for the equivalent CLI invocation.
